# Mouse DRG paclitaxel exposure: processed TPM pilot

**Question.** What descriptive transcript patterns appear in female mouse dorsal-root-ganglion (DRG) samples labeled paclitaxel versus saline control? This is an in vivo neuropathy context, not a replication of cancer-cell acute exposure.

**Sources and labels.** [GSE230454](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE230454) provides the female controls; [GSE230459](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE230459) provides the female `Pax` samples. Four columns per arm map to eight GSM records in `sample_manifest.tsv`. The [source paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC12353069/) describes paclitaxel dosing. However, the paclitaxel GEO sample `treatment_protocol` fields say saline. The labels are provisional until the submitter or raw sample metadata resolves this contradiction.

**Method.** Download two processed TPM workbooks, verify compressed-file SHA-256, select female sheets, align 48,431 gene rows by identifier, inspect per-sample totals and detection, then calculate log2((mean paclitaxel TPM + 1)/(mean control TPM + 1)). The 17 markers in four panels were declared in `analysis.py`. The background includes features with mean control TPM >= 1. TPM is compositional and cannot support count-likelihood differential expression or total-RNA claims. No p-values, pathway enrichment, or target inference are produced.

**Result.** Of 14,119 background features, 53.7% have positive descriptive ratios; the median ratio is +0.020 log2 units. Cell-cycle, interferon, cholesterol-synthesis, and injury markers are mixed. This does not identify a coherent reproduced program, particularly given the label conflict, bulk-tissue cell composition, and different species, dose, and times from the cell-line examples.


In [1]:
from pathlib import Path
import importlib.util
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
module_path = ROOT / "research/transcriptomics/gse230454-459/analysis.py"
spec = importlib.util.spec_from_file_location("drg_analysis", module_path)
analysis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis)
summary = analysis.run(ROOT)
print(summary)


{'study': 'GSE230454 control versus GSE230459 paclitaxel', 'organism': 'Mus musculus', 'tissue': 'female SW mouse dorsal root ganglia', 'samples_per_arm': 4, 'feature_rows': 48431, 'source_sha256': {'control': 'f982c3e9066d2608e62e0527f1d6386fd76767ce446f677fa3012c05c4dd5f47', 'paclitaxel': 'f89bbdd95fda8eccd9d735bac7217e2acc6b83808b840af87f0e7dbeb8a1cbdb'}, 'metadata_conflict': 'GSE230459 sample titles say Pax; their treatment_protocol fields say saline. Source paper describes paclitaxel dosing. Treat exposure labels as provisional until reconciled with submitter/raw metadata.', 'interpretation': 'Processed TPM descriptive contrast; no count-aware differential expression, pathway enrichment, or target inference.'}


In [2]:
import pandas as pd
base = ROOT / "research/transcriptomics/gse230454-459/results"
print(pd.read_csv(base / "sample_qc.tsv", sep="\t").to_string(index=False))
print(pd.read_json(base / "background_summary.json", typ="series").to_string())
print(pd.read_csv(base / "marker_descriptive.tsv", sep="\t")[["panel", "gene", "log2_tpm_plus_1_ratio"]].to_string(index=False))


sample      group  total_tpm  detected_features
ConF_1    control  1000000.0              19014
ConF_2    control  1000000.0              20749
ConF_3    control  1000000.0              18679
ConF_4    control  1000000.0              19354
PacF_1 paclitaxel  1000000.0              18527
PacF_2 paclitaxel  1000000.0              20161
PacF_3 paclitaxel  1000000.0              20810
PacF_4 paclitaxel  1000000.0              20031
eligible_features_control_mean_tpm_at_least_1    14119.000000
fraction_positive                                    0.537149
fraction_negative                                    0.462851
median_log2_tpm_plus_1_ratio                         0.020365
                panel   gene  log2_tpm_plus_1_ratio
           cell_cycle  Mki67               0.417195
           cell_cycle  Top2a              -0.047469
           cell_cycle  Ccnb1               0.138989
           cell_cycle  Ube2c              -0.273576
           cell_cycle   Cdk1               0.171004
        